# 02 — Preprocessing

**Purpose**: Clean the OASIS-2 dataset, engineer features, and create the train/test split.

All logic lives in `src/preprocessing.py`. This notebook calls those functions and inspects the outputs.

**What we cover here**:
- Missing value imputation (median per group)
- Delta feature engineering (change V1 → V2)
- Ground truth label: CDR worsened at any visit after visit 2
- Stratified train/test split
- Verify split looks correct before saving

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.preprocessing import (
    impute_missing_values,
    engineer_features_and_labels,
    split_train_test,
    save_processed_splits,
)

RAW_DATA_PATH = project_root / "data" / "raw" / "oasis_longitudinal.csv"

print(f"Project root  : {project_root}")
print(f"Raw data path : {RAW_DATA_PATH}")

## 1. Load raw dataset

In [ ]:
oasis_df = pd.read_csv(RAW_DATA_PATH)
oasis_df.columns = oasis_df.columns.str.strip()
print(f"Loaded: {oasis_df.shape[0]} rows × {oasis_df.shape[1]} columns")
oasis_df.head()

## 2. Impute missing values

MMSE (2 missing) and SES (19 missing) are imputed using the median of their participant group.
We impute per group rather than overall to avoid mixing signal across Nondemented/Demented/Converted.

In [ ]:
oasis_imputed = impute_missing_values(oasis_df)

# Verify no nulls remain in MMSE and SES
print(f"Nulls remaining in MMSE : {oasis_imputed['MMSE'].isnull().sum()}")
print(f"Nulls remaining in SES  : {oasis_imputed['SES'].isnull().sum()}")

## 3. Engineer features and ground truth labels

In [ ]:
participant_features_df = engineer_features_and_labels(oasis_imputed)
participant_features_df.head()

In [ ]:
# Inspect the delta features — these are what the models will rely on most
delta_cols = ["subject_id", "participant_group", "mmse_v1", "mmse_v2", "mmse_delta_v1_v2",
              "nwbv_v1", "nwbv_v2", "nwbv_delta_v1_v2", "cdr_v1", "cdr_v2",
              "cdr_delta_v1_v2", "cdr_worsened_after_v2"]
participant_features_df[delta_cols].sort_values("participant_group")

In [ ]:
# Inspect converter rows specifically — these are the hardest to predict
converter_features = participant_features_df[
    participant_features_df["participant_group"] == "Converted"
]
print(f"Converter participants: {len(converter_features)}")
converter_features[delta_cols]

## 4. Stratified train/test split

In [ ]:
train_df, test_df = split_train_test(participant_features_df)

In [ ]:
# Verify converters appear in both sets
train_converters = (train_df["participant_group"] == "Converted").sum()
test_converters  = (test_df["participant_group"] == "Converted").sum()
print(f"Converters in train : {train_converters}")
print(f"Converters in test  : {test_converters}")

## 5. Save processed splits

In [ ]:
save_processed_splits(train_df, test_df)